## 1. Setup

In [7]:
from pathlib import Path
from datetime import date
from hashlib import sha256
from importlib import reload
import sys

import openpyxl
import pulp as pl
from IPython.display import FileLink, display

BASE_DIR = Path.cwd()
if not (BASE_DIR / "scheduling_dataset.xlsx").exists():
    BASE_DIR = BASE_DIR / "Scheduling"
BASE_DIR = BASE_DIR.resolve()
sys.path.insert(0, str(BASE_DIR))
import dashboard

reload(dashboard)

WORKBOOK = BASE_DIR / "scheduling_dataset.xlsx"
OUTPUT_DIR = BASE_DIR / "output"
workbook = openpyxl.load_workbook(WORKBOOK, data_only=True)


def rows(sheet):
    values = list(workbook[sheet].values)
    return [
        dict(zip(values[0], row))
        for row in values[1:]
        if any(v is not None for v in row)
    ]


doctors = [
    {
        "id": r["DoctorID"],
        "name": r["Name"],
        "seniority": r["Seniority"],
        "max_shifts": r["MaxShifts"],
    }
    for r in rows("Doctors")
]
shifts = [
    {"id": r["ShiftID"], "name": r["Name"], "start": r["Start"], "end": r["End"]}
    for r in rows("Shifts")
]
days = [
    {
        "day": r["Day"],
        "date": str(r["Date"])[:10],
        "weekday": r["Weekday"],
        "is_weekend": bool(r["IsWeekend"]),
        "week": r["Day"] // 7 + 1,
    }
    for r in rows("Horizon")
]
demand = [
    {"day": r["Day"], "shift_id": r["ShiftID"], "required": r["Required"]}
    for r in rows("Demand")
]
unavailable = [
    {"doctor_id": r["DoctorID"], "day": r["Day"]} for r in rows("Unavailability")
]
source = {
    "file": WORKBOOK.name,
    "sha256": sha256(WORKBOOK.read_bytes()).hexdigest(),
    "sheets": ["Doctors", "Shifts", "Horizon", "Demand", "Unavailability"],
    "classification": "Teaching dataset",
    "readme": [
        " | ".join(str(v) for v in row if v is not None)
        for row in workbook["README"].values
        if any(v is not None for v in row)
    ],
}
workbook.close()
data = {
    "doctors": doctors,
    "shifts": shifts,
    "days": days,
    "demand": demand,
    "unavailability": unavailable,
    "source": source,
}
data

{'doctors': [{'id': 'D1',
   'name': 'Dr. Adams',
   'seniority': 'Senior',
   'max_shifts': 8},
  {'id': 'D2', 'name': 'Dr. Brown', 'seniority': 'Senior', 'max_shifts': 8},
  {'id': 'D3', 'name': 'Dr. Carter', 'seniority': 'Junior', 'max_shifts': 8},
  {'id': 'D4', 'name': 'Dr. Diaz', 'seniority': 'Junior', 'max_shifts': 8},
  {'id': 'D5', 'name': 'Dr. Evans', 'seniority': 'Junior', 'max_shifts': 7},
  {'id': 'D6', 'name': 'Dr. Foster', 'seniority': 'Junior', 'max_shifts': 7}],
 'shifts': [{'id': 'Day', 'name': 'Daytime', 'start': '08:30', 'end': '16:00'},
  {'id': 'Evening', 'name': 'Evening', 'start': '16:00', 'end': '23:00'},
  {'id': 'Night', 'name': 'Night', 'start': '23:00', 'end': '08:30'}],
 'days': [{'day': 0,
   'date': '2025-11-03',
   'weekday': 'Monday',
   'is_weekend': False,
   'week': 1},
  {'day': 1,
   'date': '2025-11-04',
   'weekday': 'Tuesday',
   'is_weekend': False,
   'week': 1},
  {'day': 2,
   'date': '2025-11-05',
   'weekday': 'Wednesday',
   'is_weekend'

## 2. Constraints

**Assignment variables.** There are 6 doctors ($p\in P=\{D1,\ldots,D6\}$), 14 days ($d\in D=\{0,\ldots,13\}$), and 3 shifts ($s\in S=\{\mathrm{Day},\mathrm{Evening},\mathrm{Night}\}$).

$$x_{pds}\in\{0,1\}.$$

$x_{pds}=1$ means doctor $p$ works shift $s$ on day $d$; otherwise it is 0.

In [8]:
P = [p["id"] for p in doctors]
D = [d["day"] for d in days]
S = [s["id"] for s in shifts]
model = pl.LpProblem("Physician_Scheduling", pl.LpMinimize)
x = {
    (p, d, s): pl.LpVariable(f"x_{p}_{d}_{s}", cat=pl.LpBinary)
    for p in P
    for d in D
    for s in S
}

**Exact coverage.** Assign exactly 1 doctor to each of the 42 day–shift combinations.

$$\sum_{p\in P}x_{pds}=1\qquad\forall d\in D,\ s\in S.$$

In [9]:
for row in demand:
    model += pl.lpSum(x[p, row["day"], row["shift_id"]] for p in P) == row["required"]

**Total capacity.** The maximum numbers of shifts for D1–D6 are $C=(8,8,8,8,7,7)$.

$$\sum_{d\in D}\sum_{s\in S}x_{pds}\le C_p\qquad\forall p\in P.$$

In [10]:
for p in doctors:
    model += pl.lpSum(x[p["id"], d, s] for d in D for s in S) <= p["max_shifts"]

**Daily limit.** Each doctor can start at most 1 shift per day.

$$\sum_{s\in S}x_{pds}\le1\qquad\forall p\in P,\ d\in D.$$

The shifts are 08:30–16:00, 16:00–23:00 and 23:00–08:30 the next day. With these times, the daily limit prevents overlapping assignments; no extra minimum rest period is imposed.

In [11]:
for p in P:
    for d in D:
        model += pl.lpSum(x[p, d, s] for s in S) <= 1

**Unavailability.** These doctor–day pairs cannot be assigned any shift:

$$U=\{(D1,10),(D3,5),(D3,6),(D5,0),(D5,1),(D6,12)\}.$$

$$x_{pds}=0\qquad\forall(p,d)\in U,\ s\in S.$$

In [12]:
for row in unavailable:
    for s in S:
        model += x[row["doctor_id"], row["day"], s] == 0

**Night eligibility.** Senior doctors D1 and D2 cannot work nights.

$$x_{p,d,\mathrm{Night}}=0\qquad\forall p\in\{D1,D2\},\ d\in D.$$

In [13]:
for p in doctors:
    if p["seniority"] == "Senior":
        for d in D:
            model += x[p["id"], d, "Night"] == 0

**Fairness bounds.** Define total shifts $T_p$, night shifts $N_p$ for juniors D3–D6, and weekend shifts $W_p$ on days $D_W=\{5,6,12,13\}$.

$$T_p=\sum_{d,s}x_{pds},\qquad N_p=\sum_d x_{p,d,\mathrm{Night}},\qquad W_p=\sum_{d\in D_W,s}x_{pds}.$$

For each measure $k\in\{T,N,W\}$, bound the relevant doctors' workloads and define their spread:

$$L_k\le k_p\le H_k,\qquad\Delta_k=H_k-L_k.$$

Night bounds apply only to juniors; the other bounds apply to all 6 doctors. Weekend status uses the shift's start date.

In [14]:
night_doctors = [p for p in doctors if p["seniority"] == "Junior"]
weekend_days = [d["day"] for d in days if d["is_weekend"]]
total = {p: pl.lpSum(x[p, d, s] for d in D for s in S) for p in P}
night = {p["id"]: pl.lpSum(x[p["id"], d, "Night"] for d in D) for p in night_doctors}
weekend = {p: pl.lpSum(x[p, d, s] for d in weekend_days for s in S) for p in P}


def spread(name, counts):
    low = pl.LpVariable(f"{name}_minimum", 0, len(D), cat=pl.LpInteger)
    high = pl.LpVariable(f"{name}_maximum", 0, len(D), cat=pl.LpInteger)
    for count in counts.values():
        model.addConstraint(count >= low)
        model.addConstraint(count <= high)
    return high - low


objectives = [
    ("Total shift spread", spread("total", total)),
    ("Junior night shift spread", spread("night", night)),
    ("Weekend shift spread", spread("weekend", weekend)),
]

## 3. Solve

Minimize the three spreads in order, preserving each earlier optimum:

$$\operatorname{lexmin}(\Delta_T,\Delta_N,\Delta_W).$$

Solve with CBC, then generate `output/dashboard.html` using `dashboard.py`.

In [15]:
solver = pl.PULP_CBC_CMD(
    msg=False, timeLimit=60, threads=1, gapRel=0, options=["randomSeed 42"]
)
stages = []
for i, (name, objective) in enumerate(objectives):
    model.setObjective(objective)
    model.solve(solver)
    if model.status != pl.LpStatusOptimal or model.sol_status != pl.LpSolutionOptimal:
        raise RuntimeError(f"No proven optimal solution: {pl.LpStatus[model.status]}")
    optimum = round(pl.value(objective))
    stages.append({"name": name, "status": "Optimal", "optimum": optimum})
    model += objective == optimum, f"preserve_stage_{i+1}"
    print(f"{name}: {optimum}")

result, dashboard_path = dashboard.save_schedule(
    data, x, stages, night_doctors, OUTPUT_DIR
)
print(f"\n{len(result['assignments'])} shifts assigned; all constraint checks passed.")
for p in doctors:
    assigned = [a for a in result["assignments"] if a["doctor_id"] == p["id"]]
    print(
        f"{p['name']:12} Total: {len(assigned)}  Nights: {sum(a['shift_id']=='Night' for a in assigned)}"
        f"  Weekend: {sum(a['is_weekend'] for a in assigned)}"
    )
display(FileLink(str(dashboard_path.relative_to(Path.cwd()))))

Total shift spread: 0
Junior night shift spread: 1
Weekend shift spread: 0

42 shifts assigned; all constraint checks passed.
Dr. Adams    Total: 7  Nights: 0  Weekend: 2
Dr. Brown    Total: 7  Nights: 0  Weekend: 2
Dr. Carter   Total: 7  Nights: 3  Weekend: 2
Dr. Diaz     Total: 7  Nights: 3  Weekend: 2
Dr. Evans    Total: 7  Nights: 4  Weekend: 2
Dr. Foster   Total: 7  Nights: 4  Weekend: 2


/Users/jiaokan/Workshop/AIRM/Scheduling/output/dashboard.html